# Section 4 · Notebook 4 — Swarm Pattern: Game Level Design

Agents hand control to each other — there is no central coordinator:
- **Level Designer** — layout, environments, visuals
- **Game Designer** — playtests for fun and confusion
- **Narrative Designer** — story and atmosphere
- **Difficulty Balancer** — difficulty and pacing

The path is not decided in advance; it emerges from the work.

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from strands.models.bedrock import BedrockModel

MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-sonnet-4-6")
REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))
os.environ.setdefault("AWS_REGION", REGION)

model = BedrockModel(model_id=MODEL_ID, region_name=REGION)
print(f"Using {MODEL_ID} in {REGION}")

Using global.anthropic.claude-sonnet-4-6 in us-east-1


In [3]:
import logging

from strands import Agent
from strands.multiagent import Swarm

# Show each handoff as it happens
logging.getLogger("strands.multiagent").setLevel(logging.DEBUG)
logging.basicConfig(format="%(levelname)s | %(name)s | %(message)s", handlers=[logging.StreamHandler()])

## Specialist agents

In [4]:
level_designer = Agent(
    name="level_designer",
    model=model,
    callback_handler=None,
    system_prompt="""You are a Level Designer in a game design swarm.
Your role is to create level layouts, environments, and visual elements.
Focus ONLY on layout and visual design. Do NOT write narrative or balance difficulty.
After completing your layout, hand off to narrative_designer for story elements
and difficulty_balancer for gameplay tuning.
When receiving feedback from other agents, revise your design accordingly.""",
)

game_designer = Agent(
    name="game_designer",
    model=model,
    callback_handler=None,
    system_prompt="""You are a Game Designer in a game design swarm.
Your role is to playtest designs for fun, usability, and player confusion.
Focus ONLY on gameplay feel and player experience. Do NOT design layouts or write narrative.
When you find issues, hand off to the appropriate specialist to fix them.
When the design is ready, give the final summary and do not hand off.""",
)

narrative_designer = Agent(
    name="narrative_designer",
    model=model,
    callback_handler=None,
    system_prompt="""You are a Narrative Designer in a game design swarm.
Your role is to create story elements, character backstories, and atmospheric details.
Focus ONLY on narrative. Do NOT redesign layouts or adjust difficulty.
After adding narrative elements, hand off to difficulty_balancer for balance review
or game_designer for playtesting feedback.""",
)

difficulty_balancer = Agent(
    name="difficulty_balancer",
    model=model,
    callback_handler=None,
    system_prompt="""You are a Difficulty Balancer in a game design swarm.
Your role is to evaluate and tune difficulty progression, enemy placement, and pacing.
Focus ONLY on balance. Do NOT redesign layouts or write narrative.
When you find layout or narrative issues, hand off to the appropriate agent.
After balancing, hand off to game_designer for a final playtest review.""",
)

## Create the swarm

The swarm gives every agent a `handoff_to_agent` tool and a shared view of the work so far. For example, the game designer might call:

```python
handoff_to_agent(
    agent_name="difficulty_balancer",
    message="Patrol patterns too predictable - need more variation",
)
```

Safety limits:
- `max_handoffs` / `max_iterations` — stop runaway loops
- `repetitive_handoff_detection_window` / `repetitive_handoff_min_unique_agents` — stop two agents ping-ponging
- `execution_timeout` / `node_timeout` — cap wall-clock time (seconds)

In [5]:
swarm = Swarm(
    [level_designer, game_designer, narrative_designer, difficulty_balancer],
    entry_point=level_designer,
    max_handoffs=20,
    max_iterations=20,
    execution_timeout=1200.0,
    node_timeout=300.0,
    repetitive_handoff_detection_window=8,
    repetitive_handoff_min_unique_agents=3,
)
print("✓ Swarm created")

DEBUG | strands.multiagent.swarm | nodes=<['level_designer', 'game_designer', 'narrative_designer', 'difficulty_balancer']> | initialized swarm with nodes
DEBUG | strands.multiagent.swarm | entry_point=<level_designer> | configured entry point
DEBUG | strands.multiagent.swarm | tool_count=<1>, node_count=<4> | injected coordination tools into agents


✓ Swarm created


## Run the swarm (takes a few minutes)

In [6]:
result = swarm("""Design a stealth mission level in an abandoned space station.
The level needs to support multiple playstyles (aggressive stealth, patient stealth, and ghost runs).
Include at least 3 distinct zones with different difficulty levels,
narrative elements that explain why the station was abandoned,
and ensure the difficulty curve is appropriate for mid-game players.""")

print("=" * 80)
print("HANDOFF CHAIN:")
for node in result.node_history:
    print(f"  → {node.node_id}")
print(f"\nStatus: {result.status} | iterations: {result.execution_count} | time: {result.execution_time / 1000:.0f}s")

DEBUG | strands.multiagent.swarm | starting swarm execution
DEBUG | strands.multiagent.swarm | current_node=<level_designer> | starting swarm execution with node
DEBUG | strands.multiagent.swarm | max_handoffs=<20>, max_iterations=<20>, timeout=<1200.0>s | swarm execution config
DEBUG | strands.multiagent.swarm | current_node=<level_designer>, iteration=<1> | executing node
DEBUG | strands.multiagent.swarm | from_node=<level_designer>, to_node=<narrative_designer> | handing off from agent to agent
DEBUG | strands.multiagent.swarm | node=<level_designer> | node execution completed
DEBUG | strands.multiagent.swarm | from_node=<level_designer>, to_node=<narrative_designer> | handoff detected
DEBUG | strands.multiagent.swarm | current_node=<narrative_designer>, iteration=<2> | executing node
DEBUG | strands.multiagent.swarm | from_node=<narrative_designer>, to_node=<difficulty_balancer> | handing off from agent to agent
DEBUG | strands.multiagent.swarm | node=<narrative_designer> | node ex

HANDOFF CHAIN:
  → level_designer
  → narrative_designer
  → difficulty_balancer
  → game_designer

Status: Status.COMPLETED | iterations: 4 | time: 278s


In [7]:
from IPython.display import Markdown, display

last_node = result.node_history[-1].node_id
display(Markdown(str(result.results[last_node].result)))

# ECHO STATION — Final Playtest Review

## Game Designer's Holistic Assessment

All three specialist passes are complete. I've reviewed the full design across zones, playstyles, and the integrated narrative/difficulty system. Here is my comprehensive playtest evaluation.

---

## ✅ OVERALL VERDICT: **SHIP-READY WITH MINOR FLAGS**

Echo Station is a cohesive, well-paced stealth level that successfully serves all three playstyles at mid-game difficulty. The design achieves something genuinely rare: a ghost run that feels *lonely and tense*, a patient stealth run that earns *genuine emotional weight*, and an aggressive stealth run that feels *competent and kinetic* — each using the same physical space.

---

## ZONE-BY-ZONE PLAYTEST FEEL

### 🟡 ZONE 1 — Outer Ring (Docking & Maintenance Bay)
**Feel:** *Cold arrival, quiet wrongness*
**Difficulty:** Low-Medium | 5 guards ✅

**What works:**
- The amber/red strobes establish mood without overwhelming visibility. Players will read this as "danger present but not immediate" — exactly right for zone onboarding.
- Boot prints and sealed evacuation route (Docking Bay A with Vantara locks vs. emergency seals) are an excellent environmental contradiction. Players who notice will feel *smart*, not tutorialized.
- Guard density of 5 is correct. New players get their footing; experienced players coast and feel competent.
- The unguarded vent access in the Maintenance Corridor is the right call for ghost runners — they get a clean bypass that *rewards style* without feeling like the level forgot about them.

**Playtest flags:**
- 🚩 **Oduya audio log tension signal (18s log / 12s patrol gap):** The narrative designer confirmed this is intentional soft pressure, and the difficulty balancer flagged it correctly. However, in playtest feel, this *may* read as a design error to players who trigger it and get caught — they'll feel punished for engaging with story content. **Recommendation:** Ensure the audio log has a clear "cut early to save yourself" interrupt prompt (button hold to cancel), so the pressure signal teaches discipline rather than punishes curiosity. No handoff needed — this is an interaction design note for implementation.

- 🟢 **Nell's toolkit and Oduya's audio log** as paired breadcrumbs (one dead, one unknown) is elegant mystery setup. No changes needed.

---

### 🟡 ZONE 2 — Inner Ring (Crew Quarters & Research Labs)
**Feel:** *Preserved tragedy, creeping dread*
**Difficulty:** Medium | 9 guards ✅

**What works:**
- The jump from 5 to 9 guards reads as meaningful escalation without being punishing — this is mid-game appropriate.
- The Mess Hall's 22-second patrol cycle is the single best design decision in the level. It creates a genuine *breathing room* for the half-eaten meal and Adara's unfinished message to her daughter. Patient stealth players will stop here. This is where the level earns its emotional stakes.
- Lab B's cracked terrarium with the 5-second guard hesitation is mechanically elegant: the guard *pauses at the thing that caused the disaster*, which rewards observant players with both a gameplay window and a narrative revelation simultaneously. **This is a design highlight.**
- The ventilation network being noise-active is the correct call for ghost runners in this zone. It creates discipline pressure without introducing new mechanics — the system players already know just becomes less forgiving.

**Playtest flags:**
- 🚩 **Server Room 15-second extraction as tension peak:** This works for aggressive and patient stealth. For ghost runners, who arrive with zero resource expenditure, this may feel like an anti-climax (they've been *so careful*, and then it's just... wait 15 seconds in a room). **Recommendation:** Add a minor environmental payoff during the 15-second wait — a flickering screen with a partial Vantara transmission, or a data-slate with Adara's survey report header visible. Not a full narrative beat — just enough to reward the ghost runner's clean approach with a *glimpse* of Zone 3's revelations. Implementation note only, no handoff needed.

- 🚩 **Lab B hesitation as fixed behavior (not ambient flavor):** The difficulty balancer correctly flagged this as a patrol timing dependency. If this guard hesitation is ever treated as flavor and varied or removed, the 5-second window disappears and Lab B becomes the hardest room in Zone 2 unexpectedly. This **must** be locked as fixed behavior in implementation. Implementation note only.

- 🟢 **Ventilation network as both ghost-run route and Bloom spore history** is excellent layered design. No changes needed.

---

### 🟡 ZONE 3 — The Core (Command Center & Reactor Chamber)
**Feel:** *Institutional betrayal, earned grief*
**Difficulty:** High | 9 guards (higher complexity) ✅

**What works:**
- The irregular Collapsed Corridor timing (7–15s variance) is the correct way to signal "rules have changed" at Zone 3 entry. Players who pattern-read through Zone 2 will be briefly destabilized — this is good mid-game challenge design.
- **The spinning chair visible for 8 seconds before mid-room sweep** is the single best narrative/gameplay integration moment in the level. Eight seconds is *just long enough* for players to register the chair, feel something, and then have a guard pull their attention. The tension between "I want to stay and look at this" and "I need to move" IS the emotional experience of this mission. Do not shorten this window.
- The blast door guard facing the console (not the entry) so blast marks are always seen first is clean, purposeful environmental staging. Every playthrough type gets this beat.
- **Final Protocol as the emotional climax for patient stealth** is correctly designed as the hardest clearance in the level. The difficulty is *justified by what the moment requires* — you need the room to be still for a moral choice to land. This is sophisticated design.
- Quarantine Zone observation window visible during the timed door wait on the Emergency Access Shaft is a masterclass in ghost-run storytelling: the figures in blue-white light, no context, no explanation. Ghost runners *see* but don't *understand*. This is the correct level of emotional engagement for a playstyle that chose speed over story.

**Playtest flags:**
- 🚩 **Reactor Chamber triangle formation (flag for 2s rotation offset):** The difficulty balancer flagged this correctly — if the triangle is cleanly chainable without gadgets, it trivializes a core Zone 3 challenge for aggressive stealth players. **Recommendation:** Confirm this flag is resolved before shipping. If playtesting shows clean chains, implement the 2-second rotation offset. This needs a final implementation pass but no specialist rework.

- 🚩 **Emergency Access Shaft timed door (10s open / 30s recharge):** The 10-second window is tight but fair. The 30-second recharge creates genuine cost for missed timing. **One concern:** If a player triggers the door, misses it, and must wait 30 seconds in the shaft — is there environmental content to fill that wait? The Quarantine observation window solves this *if* it's visible during the wait. **Confirm the window is reachable during the full 30-second recharge, not just during the 10-second open.** This is the difference between a tense wait and a dead wait. Implementation note only.

- 🟢 **Morrow's final handwritten note ("Someone will come. Tell them we tried.")** as the emotional endpoint is the right call. No changes needed. It works for every playstyle that reaches it.

---

## PLAYSTYLE ARCS — HOLISTIC ASSESSMENT

### 👻 Ghost Run
**Arc:** Low → Medium (vent discipline) → Sharp spike (timed door) → Emotional resolution (Quarantine figures)
**Verdict:** ✅ **Excellent.** The ghost run is lonely in the best way. The vent network keeps the player physically separate from the horror they're moving through, and the Quarantine Zone payoff gives them an image they can't explain. The unguarded Zone 1 vent access, noise-active Zone 2 vents, and Zone 3 timed door create a clean mechanical escalation. Ghost runners are rewarded with *the haunting image* — the six figures in blue-white light — and nothing else. That's correct.

### 🕵️ Patient Stealth
**Arc:** Low → Sustained medium → High → Quiet climax (Final Protocol)
**Verdict:** ✅ **Excellent.** This is the "full game" playthrough. Every environmental hook lands, every named crew member gets their moment, and Final Protocol is earned by players who *chose* to understand what happened. The Mess Hall dwell time, Adara's unfinished message, Morrow's personal chair — these accumulate correctly. The emotional weight is proportional to investment.

### 💥 Aggressive Stealth
**Arc:** Low → Escalating → Demanding → Cathartic (Command Center clearance)
**Verdict:** ✅ **Good, with one note.** The Command Center clearance as cathartic endpoint works well. The Reactor Chamber triangle formation is the correct mechanical peak. **One concern:** Aggressive stealth players may bypass enough Zone 2 narrative that Zone 3's Quarantine Zone reveal lands without sufficient setup — they'll see the figures but may lack context for *why* they're in stasis (Adara's research logs are in Zone 2, which aggressive players may not dwell in). This isn't a flaw — it's a *design choice* — but it should be *intentional*. If the goal is that aggressive stealth players feel confused by the Quarantine Zone, that's valid and evocative. If the goal is that they understand enough to feel the horror, Zone 2's audio logs in high-traffic areas need to be confirmed as unavoidable ambient (playing as players pass through, not requiring interaction). **Recommendation:** Confirm Zone 2 audio logs for aggressive stealth are ambient-triggered on patrol routes, not interaction-triggered on detour routes.

---

## DIFFICULTY CURVE ASSESSMENT

```
Zone 1 (5 guards, low-medium)  →  Zone 2 (9 guards, medium)  →  Zone 3 (9 guards, high complexity)
[LEARNING]                          [APPLICATION]                    [MASTERY TEST]
```

**Verdict:** ✅ **Appropriate for mid-game.** The curve is not a simple numerical escalation — Zone 3 matches Zone 2 in guard count but increases *cognitive load* through irregular timing and formation complexity. This is the correct design for mid-game players who've internalized basic stealth patterns and need to be challenged on *reading and adapting*, not just reacting. A new player would struggle at Zone 3; a late-game player would find it routine. Mid-game players will feel competent but tested. This is correct.

---

## NARRATIVE INTEGRATION ASSESSMENT

The Kessler Bloom as a *non-violent* organism is the level's best thematic decision. The horror isn't the creature — it's Vantara. The creature is almost peaceful. This inversion (corporation as monster, organism as victim alongside crew) gives the level genuine moral texture that rewards players who engage. The three-act escalation (mystery → tragedy → betrayal) maps cleanly to the three zones without feeling mechanical.

**Morrow's refusal to enter Final Protocol** is the emotional core of the level, and it's correctly positioned as the *last thing* patient stealth players discover. The level earns this moment.

---

## FINAL IMPLEMENTATION FLAGS (No Handoffs Required)

| # | Flag | Zone | Priority |
|---|------|-------|----------|
| 1 | Oduya audio log must have an early-cancel interrupt prompt | Zone 1 | Medium |
| 2 | Server Room 15s wait — add ambient data screen glimpse for ghost runners | Zone 2 | Low |
| 3 | Lab B guard hesitation (5s) must be locked as fixed patrol behavior, never ambient flavor | Zone 2 | **High** |
| 4 | Reactor Chamber triangle formation: test for clean chain; apply 2s offset if cleanly chainable without gadgets | Zone 3 | **High** |
| 5 | Confirm Quarantine Zone window is visible during full 30s timed door recharge (not just 10s open) | Zone 3 | **High** |
| 6 | Confirm Zone 2 audio logs trigger as ambient (on patrol proximity) for aggressive stealth playthroughs | Zone 2 | Medium |

---

## SUMMARY

**ECHO STATION is a complete, ship-ready stealth level.**

It succeeds on every core requirement:
- ✅ Three distinct zones with coherent visual and tonal identities
- ✅ Three playstyles with meaningfully different experiences of the *same physical space*
- ✅ Difficulty curve calibrated for mid-game players — escalating, fair, and never arbitrary
- ✅ Narrative that rewards engagement without punishing speed
- ✅ A mystery that resolves into genuine moral weight for players who seek it
- ✅ Environmental storytelling dense enough to carry the story without a single line of dialogue required

The level's defining achievement is **the spinning chair** — eight seconds, before a guard pulls your attention away. In those eight seconds, every playstyle gets something: ghost runners get an image, patient stealth players get a question, aggressive stealth players get a target to clear. The whole level is in that moment.

**Echo Station is ready.**


## Key takeaways

1. **Handoffs, not routing** — agents pass control to each other
2. **Shared context** — every agent sees what has been done so far
3. **Emergent workflow** — the sequence depends on the issues that come up
4. **Safety limits** stop infinite loops
5. **Use it for** creative and iterative work; **use a graph** when you need guaranteed steps